# 02 — Device & Instance Management

This notebook covers how to discover, select, and connect to Ivium devices — including scenarios with multiple IviumSoft instances running simultaneously.

### Concepts

- **IviumSoft instance**: one running copy of the IviumSoft application. Up to 32 can run at once.
- **Device / channel**: the physical potentiostat connected to a given instance.
- **Serial number**: the unique identifier printed on each Ivium hardware unit.

```
 IviumSoft instance 1  ──►  CompactStat #A  (SN: 12345)
 IviumSoft instance 2  ──►  CompactStat #B  (SN: 67890)
 IviumSoft instance 3  ──►  (no device)
```

Pyvium talks to **one instance at a time**. You switch between them with `select_iviumsoft_instance()`.

In [ ]:
from pyvium import Pyvium
from pyvium.errors import (
    IviumSoftNotRunningError,
    DeviceNotConnectedToIviumSoftError,
    NoDeviceDetectedError,
    InvalidStateError,
)

## 1. Discovering Running IviumSoft Instances

In [ ]:
Pyvium.open_driver()

instances = Pyvium.get_active_iviumsoft_instances()
max_devices = Pyvium.get_max_device_number()

print(f"Active IviumSoft instances : {instances}")
print(f"Max manageable devices     : {max_devices}")

## 2. Selecting an Instance

All subsequent calls operate on the **currently selected** instance. If you only have one IviumSoft open, this step is optional — instance 1 is selected by default.

In [ ]:
# Select instance 1 (the first running IviumSoft)
Pyvium.select_iviumsoft_instance(1)

status_code, status_label = Pyvium.get_device_status()
print(f"Instance 1 — status: {status_code} ('{status_label}')")

# Status codes:
#  -1 : no IviumSoft on this slot
#   0 : IviumSoft running but device not connected
#   1 : device connected and idle
#   2 : device connected and busy (running a method)
#   3 : no device detected on USB

## 3. Iterating All Instances

When running multiple devices in parallel, loop over all active instances to get their status.

In [ ]:
instances = Pyvium.get_active_iviumsoft_instances()

for instance_number in instances:
    Pyvium.select_iviumsoft_instance(instance_number)
    code, label = Pyvium.get_device_status()
    print(f"  Instance {instance_number}: {label}")

## 4. Connecting a Device

"Connecting" tells IviumSoft to take control of the physical device. The device must be detected on USB first (status != 3).

> **Side effect:** `connect_device()` always resets **Auto E-ranging to off** on the instrument to establish a known state. If you rely on Auto E-ranging being active, re-enable it explicitly after connecting with `set_bistat_mode(1)` (see notebook `05_bipotentiostat_and_we32.ipynb`).

In [ ]:
Pyvium.select_iviumsoft_instance(1)

code, label = Pyvium.get_device_status()
print(f"Before connect: {label}")

if code == 0:  # IviumSoft running but no device connected yet
    try:
        Pyvium.connect_device()
        code, label = Pyvium.get_device_status()
        print(f"After connect : {label}")
    except InvalidStateError:
        print("No device available — all devices may be claimed by other instances")
    except NoDeviceDetectedError:
        print("No device detected — check USB cable")
elif code == 1:
    print("Device already connected and idle")
elif code == 2:
    print("Device connected and busy")
elif code == 3:
    print("No device detected — check USB cable")

## 5. Reading the Serial Number

The serial number uniquely identifies each physical device. Useful for confirming which device is active when multiple units are in use.

In [ ]:
try:
    serial = Pyvium.get_device_serial_number()
    print(f"Device serial number: {serial}")
except DeviceNotConnectedToIviumSoftError:
    print("Device not connected — connect it first")

## 6. Selecting a Device by Serial Number

`select_serial_number()` is the recommended way to target a **specific physical device** regardless of which IviumSoft slot it appears in. This is more robust than relying on slot numbers, which can shift when devices are unplugged and reconnected.

Returns the 0-based index of the device in the IviumSoft dropdown.

In [ ]:
TARGET_SERIAL = "12345"  # Replace with your device's serial number

try:
    index = Pyvium.select_serial_number(TARGET_SERIAL)
    print(f"Device '{TARGET_SERIAL}' found at dropdown index {index}")

    Pyvium.connect_device()
    print("Device connected")

except DeviceNotConnectedToIviumSoftError as e:
    print(f"Could not select device: {e}")

## 7. Multichannel: Selecting a Channel

Ivium-n-Soft (the multichannel variant) manages multiple channels within one IviumSoft instance. `select_channel()` opens the tab and makes that channel active.

This is **different from** `select_iviumsoft_instance()` — channel selection is within a single IviumSoft window, while instance selection switches between separate IviumSoft windows.

In [ ]:
CHANNEL = 1  # Channels are 1-indexed

Pyvium.select_channel(CHANNEL)
print(f"Channel {CHANNEL} selected")

# Now you can connect and control the device on this channel
Pyvium.connect_device()
print("Device on channel connected")

## 8. Disconnecting

Call `disconnect_device()` to release IviumSoft's control of the device without closing IviumSoft. Useful in multi-device workflows where ownership needs to be handed off.

In [ ]:
Pyvium.disconnect_device()
print("Device disconnected")

code, label = Pyvium.get_device_status()
print(f"Status after disconnect: {label}")

## 9. Complete Multi-Instance Workflow

Template for iterating all active instances, connecting each one, doing something, then moving to the next.

In [ ]:
import time

try:
    Pyvium.open_driver()
    instances = Pyvium.get_active_iviumsoft_instances()
    print(f"Found {len(instances)} active instance(s): {instances}")

    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        code, label = Pyvium.get_device_status()
        print(f"\nInstance {inst}: {label}")

        if code == 0:
            Pyvium.connect_device()
            serial = Pyvium.get_device_serial_number()
            print(f"  Connected — serial: {serial}")

            # ... do work on this device ...

            Pyvium.disconnect_device()
            print(f"  Disconnected")

        elif code == 1:
            serial = Pyvium.get_device_serial_number()
            print(f"  Already connected — serial: {serial}")

        elif code == 2:
            print(f"  Busy — skipping")

        elif code == 3:
            print(f"  No device on USB — skipping")

finally:
    Pyvium.close_driver()

---

## Summary

| Task | Method |
|------|--------|
| List running IviumSoft windows | `get_active_iviumsoft_instances()` |
| Switch between IviumSoft windows | `select_iviumsoft_instance(n)` |
| Get device state | `get_device_status()` → `(code, label)` |
| Connect selected device | `connect_device()` |
| Disconnect device | `disconnect_device()` |
| Get serial number | `get_device_serial_number()` |
| Select device by serial number | `select_serial_number(sn)` |
| Select multichannel tab | `select_channel(n)` |

## Next

- **`03_direct_mode_basics.ipynb`** — Set potential, read current, control the cell directly